# Kickstarter Campaign Success Prediction and Segmentation

**Portfolio adaptation of the INSY 446 individual project.**

1. **Classification:** Gradient Boosting predicts `successful` versus `failed` using engineered campaign attributes. The original result was assessed with a **random holdout**; it has not been validated on future time periods.
2. **Retrospective clustering:** K-Means groups historical campaigns using observed, post-launch attributes, **including the true outcome (`state`)**. These clusters are not suitable for pre-launch scoring as implemented.

**Dataset:** Download the original, appropriately licensed `Kickstarter.xlsx` and place it in `data/`. It is not distributed in this repository.

**Reproducibility:** The original reported accuracy was **75.73%**. A fresh run of the original fitted model on the provided workbook produced **75.80%** in the current Python/scikit-learn environment. Results may differ slightly by library version. See `VALIDATION.md`.

**Computation:** The full GridSearchCV has 360 model fits and is disabled by default to make this notebook easier to run on a laptop. Set `RUN_GRID_SEARCH = True` if you want to repeat it. Silhouette scores use reproducible sampling to avoid very large pairwise-distance matrices.

## Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as ss
from pathlib import Path


In [ ]:
# Locate data whether the kernel starts in the repository root or notebooks/.
from pathlib import Path
DATA_CANDIDATES = [
    Path("data/Kickstarter.xlsx"),
    Path("../data/Kickstarter.xlsx"),
]
DATA_FILE = next((p for p in DATA_CANDIDATES if p.is_file()), None)
if DATA_FILE is None:
    raise FileNotFoundError(
        "Kickstarter.xlsx not found. Put it under data/ and run from "
        "the repository root or the notebooks/ directory."
    )
print("Using dataset:", DATA_FILE.resolve())

In [ ]:
first_d1f = pd.read_excel(DATA_FILE)

In [ ]:
with pd.option_context('display.max_columns', None):
    display(first_d1f.head())


In [ ]:
first_d1f.info()

In [ ]:
first_d1f.describe()

## Task 1 - Classification

In [ ]:
# initial variable selection

# drop columns that are useless to the model and columns that contain leaking informations
first_d1f = first_d1f.drop(['spotlight','pledged','usd_pledged','backers_count','staff_pick', 'staff_pick.1','disable_communication'], axis = 1)


# drop columns that are redundant (day, month, day and time),
first_d1f = first_d1f.drop(columns=[
    # deadline
    'deadline_weekday',
    'deadline_month', 'deadline_day', 'deadline_yr', 'deadline_hr',

    # state_changed_at
    'state_changed_at_weekday',
    'state_changed_at_month', 'state_changed_at_day',
    'state_changed_at_yr', 'state_changed_at_hr',

    # created_at
    'created_at_weekday',
    'created_at_month', 'created_at_day',
    'created_at_yr', 'created_at_hr',

    # launched_at
    'launched_at_month', 'launched_at_day',
    'launched_at_yr', 'launched_at_hr'
])

first_d1f = first_d1f.drop(['id','name'], axis = 1)



In [ ]:
# only use successful or failed state
first_d1f = first_d1f[first_d1f['state'].isin(['successful', 'failed'])]

In [ ]:
# Removing duplicates
first_d1f.duplicated().sum()

In [ ]:
# check unary
nearly_unary = []

for col in first_d1f.columns:
    top_ratio = first_d1f[col].value_counts(normalize=True, dropna=False).iloc[0]
    if top_ratio > 0.98:
        nearly_unary.append((col, top_ratio))

nearly_unary

In [ ]:
# check null
with pd.option_context('display.max_columns', None):
    display(first_d1f[first_d1f.isnull().any(axis=1)].head())

In [ ]:
# filling null
first_d1f['main_category'] = first_d1f['main_category'].fillna(first_d1f['category'])

In [ ]:
# check null
with pd.option_context('display.max_columns', None):
    display(first_d1f[first_d1f.isnull().any(axis=1)].head())

In [ ]:
# check for correlation for text_based variables
def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = ss.chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    return np.sqrt((chi2 / n) / (min(r, k) - 1))

cv_cc = cramers_v(first_d1f['country'], first_d1f['currency'])
cv_cat = cramers_v(first_d1f['category'], first_d1f['main_category'])
print("Cramér's V:", round(cv_cc, 4))
print("Cramér's V:", round(cv_cat, 4))



In [ ]:
# drop columns that are redundant with another
first_d1f = first_d1f.drop(['category','currency'], axis = 1)

In [ ]:
# check correlation
numeric_first_d1f = first_d1f.select_dtypes(include=['int64', 'float64'])

plt.figure(figsize=(6,5))
sns.heatmap(numeric_first_d1f.corr(), annot=False, cmap="coolwarm")
plt.show()


In [ ]:
# drop columns that are redundant with another
first_d1f = first_d1f.drop(['name_len','blurb_len'],axis = 1)

In [ ]:
try:
    import missingno as msno
    msno.matrix(first_d1f)
except ImportError:
    print("Optional visualization skipped: install missingno to display missing-data matrix.")

In [ ]:
# convert timestamp to durations and drop the timestamp
first_d1f['duration_days'] = (first_d1f['deadline'] - first_d1f['launched_at']).dt.total_seconds() / 86400 #funding duration
first_d1f['lead_time_days'] = (first_d1f['launched_at'] - first_d1f['created_at']).dt.total_seconds() / 86400 #prepare time
first_d1f = first_d1f.drop(['deadline', 'launched_at','created_at', 'state_changed_at'], axis = 1)

# create a new variable to specifically check if a project is launched at the weekend
first_d1f['launchatweekend'] = first_d1f['launched_at_weekday'].isin(['Saturday','Sunday']).astype(int)
first_d1f = first_d1f.drop(['launched_at_weekday'], axis = 1)

In [ ]:
# 1. Count how many rows in each country
country_counts = first_d1f['country'].value_counts()
with pd.option_context('display.max_rows', None):
    display(country_counts)


In [ ]:
threshold = 50

# count the number in each country
country_counts = first_d1f['country'].value_counts()

# select the major (>50) countries
major_countries = country_counts[country_counts >= threshold].index

# delected other countries and replaced with "other"
first_d1f['country_clean'] = first_d1f['country'].where(first_d1f['country'].isin(major_countries), 'Other')
first_d1f=first_d1f.drop(['country'], axis = 1)


In [ ]:
# convert boolean to numbers
first_d1f['video'] = first_d1f['video'].astype(int)
first_d1f['show_feature_image'] = first_d1f['show_feature_image'].astype(int)
first_d1f['state'] = first_d1f['state'].map({'successful': 1, 'failed': 0})

In [ ]:
# Convert the goal amount into USD using the statis currency conversion rate
first_d1f['goal_usd'] = first_d1f['goal']*first_d1f['static_usd_rate']
first_d1f['goal_usd'].skew()

In [ ]:
# Apply log(1 + x) transformation to compress extremely large goal values 
# and reduce the strong right-skewness in the original goal_usd distribution.
first_d1f['log_goal_usd'] = np.log1p(first_d1f['goal_usd'])
first_d1f['log_goal_usd'].hist(bins=50)
plt.xlabel('log_goal_usd')
plt.ylabel('Frequency')
plt.title('Distribution of log_goal_usd')
plt.show()

print(first_d1f['log_goal_usd'].skew())

In [ ]:
# convert to log
first_d1f['goal'] = first_d1f['log_goal_usd']
first_d1f = first_d1f.drop(['static_usd_rate', 'goal_usd','log_goal_usd'], axis = 1)

In [ ]:
# dummify
first_d1f = pd.get_dummies(first_d1f, 
                    columns=['main_category', 'country_clean'], 
                    drop_first=True,
                    dtype='int')

In [ ]:
with pd.option_context('display.max_columns', None):
    display(first_d1f.head())

#### Model

In [ ]:
X = first_d1f.drop(['state'], axis = 1)
y = first_d1f['state']

In [ ]:
# Separate the data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 114514)

# Standardize the data
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
scaled_X_train = scaler.fit_transform(X_train)
scaled_X_test = scaler.transform(X_test)

scaled_X_train = pd.DataFrame(scaled_X_train, columns=X_train.columns)
scaled_X_test = pd.DataFrame(scaled_X_test, columns=X_test.columns)

In [ ]:
# Lasso Feature selection
from sklearn.linear_model import Lasso
ls = Lasso(alpha = 0.005)
model = ls.fit(scaled_X_train, y_train)


lasso_result = pd.DataFrame(list(zip(X_train.columns,model.coef_)), columns = ['predictor','coefficient'])
lasso_columns = lasso_result.loc[lasso_result['coefficient'] != 0, 'predictor']
lasso_columns.count()

In [ ]:
lasso_columns

In [ ]:
Xtr = scaled_X_train.loc[:, lasso_columns]
Xte = scaled_X_test.loc[:, lasso_columns]

In [ ]:
# Hyperparameter tuning (optional: 72 combinations × 5 folds = 360 fits)
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

RUN_GRID_SEARCH = False
param_grid = {
    "n_estimators": [250, 300, 400, 550],
    "learning_rate": [0.05, 0.08, 0.1],
    "max_depth": [2, 3, 4],
    "subsample": [0.7, 0.8],
}
grid = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=0),
    param_grid=param_grid, cv=5, scoring="accuracy",
    n_jobs=2, verbose=1,
)

In [ ]:
if RUN_GRID_SEARCH:
    # WARNING: feature selection was performed before CV in the original analysis.
    # For a fully unbiased selection/tuning CV, move it into an sklearn Pipeline.
    grid.fit(Xtr, y_train)
    print("Best CV score:", grid.best_score_)
    print("Best params:", grid.best_params_)
else:
    print("Grid search skipped (RUN_GRID_SEARCH=False).")
    print("Original best parameters: learning_rate=0.1, max_depth=2, n_estimators=550, subsample=0.8")

In [ ]:
# Final Model!
gbt = GradientBoostingClassifier(random_state = 7, learning_rate= 0.1, max_depth= 2, n_estimators= 550, subsample =0.8)
model_gbt = gbt.fit(Xtr, y_train)
y_test_pred = model_gbt.predict(Xte)
acc = accuracy_score(y_test, y_test_pred)
print(acc)

## Task 2 - Clustering

In [ ]:
second_d2f = pd.read_excel(DATA_FILE)

In [ ]:
with pd.option_context('display.max_columns', None):
    display(second_d2f.head())


#### Basicly redo the cleaning, same as the Task 1 processes

In [ ]:
# drop columns that are redundant (day, month, day and time),
second_d2f = second_d2f.drop(columns=[
    # deadline
    'deadline_weekday',
    'deadline_month', 'deadline_day', 'deadline_yr', 'deadline_hr',

    # state_changed_at
    'state_changed_at_weekday',
    'state_changed_at_month', 'state_changed_at_day',
    'state_changed_at_yr', 'state_changed_at_hr',

    # created_at
    'created_at_weekday',
    'created_at_month', 'created_at_day',
    'created_at_yr', 'created_at_hr',

    # launched_at
    'launched_at_month', 'launched_at_day',
    'launched_at_yr', 'launched_at_hr'
])

second_d2f = second_d2f.drop(['name'], axis = 1)



In [ ]:
# only use successful or failed state
second_d2f = second_d2f[second_d2f['state'].isin(['successful', 'failed'])]

In [ ]:
# Removing duplicates
second_d2f.duplicated().sum()

In [ ]:
# check unary
nearly_unary = []

for col in second_d2f.columns:
    top_ratio = second_d2f[col].value_counts(normalize=True, dropna=False).iloc[0]
    if top_ratio > 0.98:
        nearly_unary.append((col, top_ratio))

nearly_unary

In [ ]:
second_d2f = second_d2f.drop(['disable_communication','staff_pick.1'], axis = 1)

In [ ]:
# check null
with pd.option_context('display.max_columns', None):
    display(second_d2f[second_d2f.isnull().any(axis=1)].head())

In [ ]:
# filling null
second_d2f['main_category'] = second_d2f['main_category'].fillna(second_d2f['category'])

In [ ]:
# check null
with pd.option_context('display.max_columns', None):
    display(second_d2f[second_d2f.isnull().any(axis=1)].head())

In [ ]:
# drop columns that are redundant with another
second_d2f = second_d2f.drop(['category','currency'], axis = 1)
# drop columns that are redundant with another
second_d2f = second_d2f.drop(['name_len','blurb_len'],axis = 1)

In [ ]:
# convert timestamp to durations and drop the timestamp
second_d2f['duration_days'] = (second_d2f['deadline'] - second_d2f['launched_at']).dt.total_seconds() / 86400 #funding duration
second_d2f['lead_time_days'] = (second_d2f['launched_at'] - second_d2f['created_at']).dt.total_seconds() / 86400 #prepare time
second_d2f = second_d2f.drop(['deadline', 'launched_at','created_at', 'state_changed_at'], axis = 1)

# create a new variable to specifically check if a project is launched at the weekend
second_d2f['launchatweekend'] = second_d2f['launched_at_weekday'].isin(['Saturday','Sunday']).astype(int)
second_d2f = second_d2f.drop(['launched_at_weekday'], axis = 1)

In [ ]:
# new threshold to reduce dummy variable
threshold = 150

# count the number in each country
country_counts = second_d2f['country'].value_counts()

# select the major (>100) countries
major_countries = country_counts[country_counts >= threshold].index

# delected other countries and replaced with "other"
second_d2f['country_clean'] = second_d2f['country'].where(second_d2f['country'].isin(major_countries), 'Other')
second_d2f =second_d2f.drop(['country'], axis = 1)


In [ ]:
# convert boolean to numbers
second_d2f['video'] = second_d2f['video'].astype(int)
second_d2f['show_feature_image'] = second_d2f['show_feature_image'].astype(int)
second_d2f['state'] = second_d2f['state'].map({'successful': 1, 'failed': 0})

In [ ]:
# Convert the goal amount into USD using the statis currency conversion rate
second_d2f['goal_usd'] = second_d2f['goal']*second_d2f['static_usd_rate']
second_d2f = second_d2f.drop(['static_usd_rate', 'goal', 'pledged'], axis = 1)

In [ ]:
# dummify
second_d2f = pd.get_dummies(second_d2f, 
                    columns=['main_category', 'country_clean'], 
                    drop_first=True,
                    dtype='int')

In [ ]:
with pd.option_context('display.max_columns', None):
    display(second_d2f.head())

#### Model

In [ ]:
X2 = second_d2f.drop(['id'], axis = 1)

In [ ]:
# delete unary columns to solidate our result
nearly_unary2 = []

for col in X2.columns:
    top_ratio = X2[col].value_counts(normalize=True, dropna=False).iloc[0]
    if top_ratio > 0.95:
        nearly_unary2.append((col, top_ratio))

nearly_unary2

In [ ]:
cols_to_drop = [col for col, ratio in nearly_unary2]
X2 = X2.drop(columns=cols_to_drop)


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler2 = StandardScaler()
X2_std = scaler2.fit_transform(X2)

In [ ]:
# Ps-F
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score
for i in range (2,8):    
    kmeans = KMeans(n_clusters=i, random_state = 11443)
    model = kmeans.fit(X2_std)
    labels = model.labels_
    print('K=',i,':',calinski_harabasz_score(X2_std, labels))

In [ ]:
#  elbow 
withinss = []
for i in range(2,8):
    kmeans = KMeans(n_clusters = i, random_state = 11443)
    model = kmeans.fit(X2_std)
    withinss.append(model.inertia_)

plt.plot([2,3,4,5,6,7],withinss)

In [ ]:
# Sample-based silhouette scoring avoids large O(n^2) memory usage.
from sklearn.metrics import silhouette_score
for i in range(2, 8):
    candidate = KMeans(n_clusters=i, random_state=11443)
    candidate_labels = candidate.fit_predict(X2_std)
    score = silhouette_score(
        X2_std, candidate_labels, sample_size=1500, random_state=42
    )
    print(i, ":", round(score, 4), "(sample of 1,500 rows)")

In [ ]:
# Final model!
kmeans = KMeans(n_clusters=4, random_state = 11443)
model = kmeans.fit(X2_std)

labels_kmeans = model.predict(X2_std)

In [ ]:
from sklearn.metrics import silhouette_samples

# The original code mistakenly used labels from the final k-search loop (k=7).
# Use the selected k=4 labels for a deterministic 1,500-row sample.
rng = np.random.default_rng(42)
sample_indices = np.sort(rng.choice(len(X2_std), size=min(1500, len(X2_std)), replace=False))
silhouette_values = silhouette_samples(X2_std[sample_indices], labels_kmeans[sample_indices])
silhouette_df = pd.DataFrame({
    "id": second_d2f["id"].to_numpy()[sample_indices],
    "label": labels_kmeans[sample_indices],
    "silhouette": silhouette_values,
})
for i in range(4):
    scores = silhouette_df.loc[silhouette_df["label"] == i, "silhouette"]
    print("Sample mean silhouette for cluster", i, ":", round(scores.mean(), 4))

In [ ]:
cluster_kmeans = pd.DataFrame(list(zip(second_d2f['id'],np.transpose(labels_kmeans))), columns = ['ID','Cluster label'])
cluster_kmeans

In [ ]:
cluster_counts = cluster_kmeans['Cluster label'].value_counts()
print("\nNumber of projects in each cluster (K-Means Clustering):")
print(cluster_counts)

In [ ]:
centers= pd.DataFrame(scaler2.inverse_transform(kmeans.cluster_centers_),
                           columns=X2.columns)

In [ ]:
centers_clean = centers.copy()
centers_clean[abs(centers_clean) < 1e-8] = 0  
centers_clean = centers_clean.round(3)

with pd.option_context("display.max_columns", None, "display.width", None):
    print(centers_clean)
